# INF 201 - Deliverable_2
## Introduction
This notebook contains Python code and explanations.

## G2_2 Filmon Tsegay Kahsay and Amin Elanbri
Link to github:

In [ ]:
import re
from pathlib import Path 

## Taks 1: File ingestion and regular expressions 

In [ ]:
def read_log_file(file_path: Path | str) -> tuple[str, str]: 
    """Read a text file that is encoded with either UTF-8 or ISO-8859-1

    Args: 
        file_path: Path to the file that will be read 
    
    Returns: 
        A tuple with the text content of the file and the encoding used. 
    """
    file_path = Path(file_path) # I make sure that I have a Path object 
    raw_bytes = file_path.read_bytes() #reads the whole file as bytes 

    try: # try UTF-8 first 
        text = raw_bytes.decode("utf-8") #here i turn the bytes into text 
        encoding_used = "utf-8" #remember wich encoding worked 
    except UnicodeDecodeError: # the bytes are not valid UTF-8
        text = raw_bytes.decode("iso-8859-1") # here i use Latin-1 instead 
        encoding_used = "iso-8859-1" #remembers wich encoding that worked 
    
    return text, encoding_used #returns both as tuples 

In [ ]:
# Regular expression for one valid sales line, with one named group per field 
LOG_LINE_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\{2} \d{2}:\d{2:\d{2}})\]"
    r"\[(?P<register_id>\w+)\] "
    r"item=(?P<price>\d+\.\d+)"
    r"(?P<currency>[A-Z]{3})"
    r"\((?P<status>SUCCESS|REFUND|CANCELED)\)$"
)

In [ ]:
def parse_log(text: str) -> list[dict[str, str | float]]:
    """Extract all valid sales transactions form the text of the lig file.

    Lines that do not match the valid line format (system noise) are ignored 

    Args: 
        text: The text content of a log file 

    Retuns: 
        List with one dict per valid transaction, with thw keys timestamp, 
        register_id, item, price, currency and status 
    """
    transactions = [] # her i make a list that will hold one dict per transaction 
    for line in text.splittlines(): #loops throug each line in the text 
        match = LOG_LINE_PATTERN.match(line.strip()) #try to match the pattern 
        if match is None: #the line is noise, not a sale line 
            continue 
        record = match.groupdict() #dict with the named groups as keys 
        record["price"] = float(record["price"]) #price from text to float 
        transactions.append(record) #adds the transaction to the list 

    return transactions #retuns all valid transactions 
